In [1]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

THU_MUC_GOC = Path("..").resolve()
THU_MUC_DAC_TRUNG = THU_MUC_GOC / "data" / "features"
THU_MUC_MODEL = THU_MUC_GOC / "model"
THU_MUC_REPORT = THU_MUC_GOC / "report"

THU_MUC_MODEL.mkdir(parents=True, exist_ok=True)
THU_MUC_REPORT.mkdir(parents=True, exist_ok=True)

X_train = np.load(THU_MUC_DAC_TRUNG / "X_train.npy")
y_train = np.load(THU_MUC_DAC_TRUNG / "y_train.npy")
print(f"[KNN] Đã nạp X_train: {X_train.shape} | y_train: {y_train.shape}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bat_dau = time.time()

pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier()),
])

param_knn = {
    "clf__n_neighbors": [3, 5, 7, 11],
    "clf__weights": ["uniform", "distance"],
    "clf__metric": ["cosine", "euclidean"],
}

grid_knn = GridSearchCV(
    pipe_knn,
    param_grid=param_knn,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
grid_knn.fit(X_train, y_train)

thoi_gian = time.time() - bat_dau
joblib.dump(grid_knn.best_estimator_, THU_MUC_MODEL / "knn_model.pkl")

ket_qua = {
    "model": "KNN",
    "best_params": grid_knn.best_params_,
    "cv_f1_macro": round(float(grid_knn.best_score_), 4),
    "train_time_sec": round(thoi_gian, 1),
}

with open(THU_MUC_REPORT / "cv_knn.json", "w", encoding="utf-8") as f:
    json.dump(ket_qua, f, ensure_ascii=False, indent=2)

print("\nHoàn tất KNN:", ket_qua)

[KNN] Đã nạp X_train: (12888, 1280) | y_train: (12888,)
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Hoàn tất KNN: {'model': 'KNN', 'best_params': {'clf__metric': 'cosine', 'clf__n_neighbors': 7, 'clf__weights': 'distance'}, 'cv_f1_macro': 0.9643, 'train_time_sec': 58.5}
